# Limpieza de Datos – Marine

## 1. Importación de paqueterías 


In [27]:
import os
import pandas as pd
from tabulate import tabulate
from unidecode import unidecode
import re
import numpy as np
from rapidfuzz import fuzz, process
from collections import defaultdict, Counter


## 2. Carga de datos

In [ ]:
#Carga de archivo
# Archivo original
archivo_original = "C:/Users/KOT14/Documents/Integral/Marine/Procesados/202510/Final/202510_Siniestros_Marine.xlsx"

# Cargar archivo Excel o CSV
df = pd.read_excel(archivo_original)   # o pd.read_csv("datos.csv")
#archivoreadme= "C:\Users\KOT14\OneDrive - Kot Insurance Company AG\Vida\Vida 2025\readme.txt"

### Extracción del nombre del archivo


In [29]:
# Extraer año y mes del nombre del archivo
nombre_base = os.path.basename(archivo_original)  # '202508_Siniestros_Vida.xlsx'
ano_mes = nombre_base[:6]                        # '202508'
ano = ano_mes[:4]                                # '2025'
mes = ano_mes[4:]                                # '08'
mes_num = int(ano_mes[4:])                        # 8 (convertido a entero)

# Diccionario de meses en español
meses = {
    1: "enero", 2: "febrero", 3: "marzo", 4: "abril", 5: "mayo", 6: "junio",
    7: "julio", 8: "agosto", 9: "septiembre", 10: "octubre", 11: "noviembre", 12: "diciembre"
}

nombre_archivo_salida = f"{ano_mes}_Siniestros_Marine_PROCESADO.xlsx"

# Carpeta y nombre de archivo de salida
#carpeta_salida = f"C:/Users/KOT14/OneDrive - Kot Insurance Company AG/06-RM - 02-Actuarial/6. DATABASE/03 Transporte, Carga & Embarcaciones/02 Bases Actuarial/{ano_mes}"
carpeta_salida2 = f"C:/Users/KOT14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/{ano_mes}"

# Crear carpeta si no existe
#os.makedirs(carpeta_salida, exist_ok=True)
os.makedirs(carpeta_salida2, exist_ok=True)

# Quitar filas completamente vacías
df = df.dropna(how="all")

#Agregar columna con mes
df["MES CARGA"] = f"{mes}/{ano}" 



## 3. Limpieza de datos

### 3.1 Reemplazo y relleno de datos


In [30]:
# ================================
# LIMPIEZA DE DATOS
# ================================
df.loc[df["INWARD POLICY N°"] == "3.6121E+12", "INWARD POLICY N°"] = "'3612100000008"
df.loc[df["INWARD POLICY N°"] == "3612100000008", "INWARD POLICY N°"] = "'3612100000008"
df.loc[df["DESCRIPTION"] == "-", "DESCRIPTION"] = "NO ESPECIFICADO"
df["INWARD POLICY N°"] = df["INWARD POLICY N°"].astype(str)

# Reemplazar valores específicos en COVER
df.loc[df["COVER"] == "CARGA", "COVER"] = "CARGA POLIETILENO"
df.loc[df["COVER"] == "Carga", "COVER"] = "CARGA POLIETILENO"
df.loc[df["COVER"] == "CARGO", "COVER"] = "CARGA POLIETILENO"
df.loc[df["COVER"] == "FletadoresPMI", "COVER"] = "RC FLETADORES(PMI)"
df.loc[df["COVER"] == "Casco y Maquinaria", "COVER"] = "CASCO Y MAQ."
df.loc[df["COVER"] == "CASCO", "COVER"] = "CASCO Y MAQ."
df.loc[df["COVER"] == "Daños a la Maquinaria", "COVER"] = "CASCO Y MAQ."
df.loc[df["COVER"] == "Fletadores RC PMI", "COVER"] = "RC FLETADORES(PMI)"
df.loc[df["COVER"] == "Pandi", "COVER"] = "PANDI"
df.loc[df["COVER"] == "RC Fletadores", "COVER"] = "RC FLETADORES"
df.loc[df["COVER"] == "Transporte", "COVER"] = "TRANSPORTE"
df.loc[df["COVER"] == "Aguas Profundas", "COVER"] = "AGUAS PROFUNDAS"

# Reemplazar valores específicos en Lob-Inward
df["LoB-Inward"] = df["LoB-Inward"].astype(str)
df.loc[df["LoB-Inward"] == "CARGA POLIETILENO", "LoB-Inward"] = "CARGO"
mask = df["LoB-Inward"].str.strip().str.upper() == "FLETADORES PMI"
df.loc[mask, "LoB-Inward"] = "RC FLETADORES(PMI)"

# Rellenar valores nulos en una columna con un valor fijo
df["COVER"] = df["COVER"].fillna("NO ESPECIFICADO")
df["COVERAGE"] = df["COVERAGE"].fillna("NO ESPECIFICADO")
df["STATUS"] = df["STATUS"].fillna("NO ESPECIFICADO")
df["REF. PEMEX"] = df["REF. PEMEX"].fillna("NO ESPECIFICADO")
df["LOCATION"] = df["LOCATION"].fillna("NO ESPECIFICADO")
df["DESCRIPTION"] = df["DESCRIPTION"].fillna("NO ESPECIFICADO")
df["LoB-Inward"] = df["LoB-Inward"].fillna(df['COVER'])
df.loc[df["LoB-Inward"] == "nan", 'LoB-Inward'] = df['COVER']

#Eliminar columnas que no necesitamos
df = df.drop(['NOTAS', 'REF. PEMEX'], axis=1)

# ================================
# DESCRIPTION
# ================================
df["DESCRIPTION"] = (
    df["DESCRIPTION"]
    .astype(str)
    .str.upper()
    .str.strip()
    .apply(lambda x: re.sub(r'[\'\"\[\]\(\)\{\}\{•}\{*}\{-}\{“}]', '', x))  # quita símbolos pero respeta Ñ/Á/É...
)

# ================================
# STATUS
# ================================
#df['STATUS ANTERIOR']=df['STATUS']
df.loc[(df["OSLR Inward"] == 0) & (df["Cumulative CLAIMS PAID"] > 0), "STATUS"] = "P"
df.loc[(df["OSLR Inward"] == 0) & (df["Cumulative CLAIMS PAID"] == 0), "STATUS"] = "C"
#df.loc[(df["OSLR Inward"] > 0) & (df["Cumulative CLAIMS PAID"] > 0), "STATUS"] = "T"  #Suposición mía


### 3.2 Normalización para Location


In [31]:
# ================================
# LOCATION
# ================================
#region LOCATION
municipios_path = r"C:/Users/KOT14/OneDrive - Kot Insurance Company AG/Códigos/Códigos Transporte/municipios_mexico.xlsx"
df_mun = pd.read_excel(municipios_path)

df_mun["mun_clean"] = df_mun["MUNICIPIO"].astype(str).apply(lambda x: unidecode(x).upper().strip())
df_mun["est_clean"] = df_mun["ESTADO"].astype(str).apply(lambda x: unidecode(x).upper().strip())
municipios_dict = dict(zip(df_mun["mun_clean"], df_mun["est_clean"]))

ciudades_path = "C:/Users/KOT14/OneDrive - Kot Insurance Company AG/Códigos/Códigos Transporte/ciudades_mexico.xlsx"  # archivo que sí subiste
df_ciu = pd.read_excel(ciudades_path)

df_ciu["city_clean"] = df_ciu["CIUDAD"].astype(str).apply(lambda x: unidecode(x).upper().strip())
df_ciu["estado_clean"] = df_ciu["ESTADO"].astype(str).apply(lambda x: unidecode(x).upper().strip())

ciudades_dict = dict(zip(df_ciu["city_clean"], df_ciu["estado_clean"]))

#autopistas
autopistas = {
    "MEX-QRO": "QUERÉTARO",
    "ARCO NORTE": "HIDALGO",
    "LA RUMOROSA": "BAJA CALIFORNIA",
    "CORDOBA": "VERACRUZ",
    "CORDOBA-VERACRUZ": "VERACRUZ",
    "TUXPAN": "VERACRUZ",
}

#estados de mx con acento
estados_con_acento = {
    "AGUASCALIENTES": "AGUASCALIENTES",
    "BAJA CALIFORNIA": "BAJA CALIFORNIA",
    "BAJA CALIFORNIA SUR": "BAJA CALIFORNIA SUR",
    "CAMPECHE": "CAMPECHE",
    "CHIAPAS": "CHIAPAS",
    "CHIHUAHUA": "CHIHUAHUA",
    "CIUDAD DE MEXICO": "CIUDAD DE MÉXICO",
    "COAHUILA": "COAHUILA",
    "COLIMA": "COLIMA",
    "DURANGO": "DURANGO",
    "ESTADO DE MEXICO": "ESTADO DE MÉXICO",
    "GUANAJUATO": "GUANAJUATO",
    "GUERRERO": "GUERRERO",
    "HIDALGO": "HIDALGO",
    "JALISCO": "JALISCO",
    "MICHOACAN": "MICHOACÁN",
    "MORELOS": "MORELOS",
    "NAYARIT": "NAYARIT",
    "NUEVO LEON": "NUEVO LEÓN",
    "OAXACA": "OAXACA",
    "PUEBLA": "PUEBLA",
    "QUERETARO": "QUERÉTARO",
    "QUINTANA ROO": "QUINTANA ROO",
    "SAN LUIS POTOSI": "SAN LUIS POTOSÍ",
    "SINALOA": "SINALOA",
    "SONORA": "SONORA",
    "TABASCO": "TABASCO",
    "TAMAULIPAS": "TAMAULIPAS",
    "TLAXCALA": "TLAXCALA",
    "VERACRUZ": "VERACRUZ",
    "YUCATAN": "YUCATÁN",
    "ZACATECAS": "ZACATECAS"
}

# abreviaturas de estados
estados_abrev = {
    "AGS": "AGUASCALIENTES",
    "BC": "BAJA CALIFORNIA",
    "BCS": "BAJA CALIFORNIA SUR",
    "CAMP": "CAMPECHE",
    "CHIS": "CHIAPAS",
    "CHIH": "CHIHUAHUA",
    "COAH": "COAHUILA",
    "COL": "COLIMA",
    "CDMX": "CIUDAD DE MÉXICO",
    "D.F.": "CIUDAD DE MÉXICO",
    "D.F": "CIUDAD DE MÉXICO",
    "DGO": "DURANGO",
    "GTO": "GUANAJUATO",
    "GRO": "GUERRERO",
    "HGO": "HIDALGO",
    "JAL": "JALISCO",
    "EDO MEX": "ESTADO DE MÉXICO",
    "EDO DE MEX": "ESTADO DE MÉXICO",
    "EDO DE MEXICO": "ESTADO DE MÉXICO",
    "EDOMEX": "ESTADO DE MÉXICO",
    "MEX": "ESTADO DE MÉXICO",
    "MICH": "MICHOACÁN",
    "MOR": "MORELOS",
    "NAY": "NAYARIT",
    "NL": "NUEVO LEÓN",
    "NL.": "NUEVO LEÓN",
    "N.L": "NUEVO LEÓN",
    "N.L.": "NUEVO LEÓN",
    "OAX": "OAXACA",
    "PUE": "PUEBLA",
    "QRO": "QUERÉTARO",
    "QROO": "QUINTANA ROO",
    "SLP": "SAN LUIS POTOSÍ",
    "SIN": "SINALOA",
    "SON": "SONORA",
    "TAB": "TABASCO",
    "TAM": "TAMAULIPAS",
    "TLAX": "TLAXCALA",
    "VER": "VERACRUZ",
    "VERACRZ": "VERACRUZ",
    "TAD PAJARITOS": "VERACRUZ",
    "YUC": "YUCATÁN",
    "ZAC": "ZACATECAS",
}

# ================================
# FUNCIONES PREVIAS A BUSCAR ESTADO
# ================================
def clean_keep_accents(texto):
    if pd.isna(texto):
        return ""
    # Mantener acentos, quitar simbolitos molestos
    texto = str(texto).upper()
    texto = re.sub(r"[^A-ZÁÉÍÓÚÜÑ0-9\s.,-]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def fuzzy_match(lista, texto, threshold=85):
    match = process.extractOne(texto, lista, scorer=fuzz.partial_ratio)
    if match and match[1] >= threshold:
        return match[0]
    return None

def load_municipios(municipios_path):
    df_mun = pd.read_excel(municipios_path)
    df_mun["MUNICIPIO_CLEAN"] = df_mun["MUNICIPIO"].str.upper()
    df_mun["ESTADO_CLEAN"] = df_mun["ESTADO"].str.upper()

    municipios_dict = dict(zip(df_mun["MUNICIPIO_CLEAN"], df_mun["ESTADO_CLEAN"]))
    estados_set = set(df_mun["ESTADO_CLEAN"])
    return municipios_dict, estados_set

# ================================
# FUNCIÓN PRINCIPAL DE MÉXICO Y USA
# ================================

usa_keywords = [
    "USA", "UNITED STATES", "EUA", "EEUU",
    "TEXAS", "HOUSTON", "GALVESTON", "BEAUMONT", "PORT ARTHUR"
]
usa_pattern = re.compile("|".join(usa_keywords), re.IGNORECASE)

# lista de estados MX para evitar falsos positivos
estados_mx = list(estados_abrev.values())  # ya lo tienes
estados_mx_pattern = re.compile("|".join(estados_mx), re.IGNORECASE)

#Estados de USA
def es_usa(texto):
    t = clean_keep_accents(texto)

    # Si contiene palabra de USA…
    if usa_pattern.search(t):
        # …pero también menciona estados mexicanos → NO es USA
        if estados_mx_pattern.search(t):
            return False
        return True
    
    return False

municipios_map = defaultdict(set)
for mun, est in zip(df_mun["mun_clean"], df_mun["est_clean"]):
    municipios_map[mun].add(est)

# Lista ordenada largo→corto
mun_list = sorted(municipios_map.keys(), key=len, reverse=True)
mun_pattern = re.compile(r"\b(" + "|".join(re.escape(m) for m in mun_list) + r")\b")

# Ciudades
ciu_list = sorted(ciudades_dict.keys(), key=len, reverse=True)
ciu_pattern = re.compile(r"\b(" + "|".join(re.escape(c) for c in ciu_list) + r")\b")

# Autopistas
auto_pattern = re.compile(r"\b(" + "|".join(re.escape(a) for a in autopistas.keys()) + r")\b")

# Abreviaturas
abrev_pattern = re.compile(r"\b(" + "|".join(re.escape(a) for a in estados_abrev.keys()) + r")\b")

# Estados MX para fuzzy simplificado
states_set = set(estados_abrev.values())

#Municipios y Estados de MX
def get_municipio_estado(texto):
    t = clean_keep_accents(texto)

    # ---------- PRIMERO: USA ----------
    if es_usa(t):
        return pd.Series([None, "USA"])
    
    # ---------- LUEGO: MEXICO ----------
    m = mun_pattern.search(t)
    if m:
        mun = m.group(1)
        estados = list(municipios_map[mun])
        if len(estados) == 1:
            return pd.Series([mun, estados[0]])
        # si hay ambigüedad → intentar encontrar el estado explícito
        for e in estados:
            if e in t:
                return pd.Series([mun, e])
        # fallback: primer estado
        return pd.Series([mun, estados[0]])

    # ---------- AUTOPISTAS ----------
    m = auto_pattern.search(t)
    if m:
        auto = m.group(1)
        return pd.Series([auto, autopistas[auto]])

    # ---------- CIUDADES ----------
    m = ciu_pattern.search(t)
    if m:
        city = m.group(1)
        return pd.Series([city, ciudades_dict[city]])

    # ---------- ABREVIATURAS ----------
    m = abrev_pattern.search(t)
    if m:
        ab = m.group(1)
        return pd.Series([None, estados_abrev[ab]])
    
    # ---------- BUSCAR ESTADO (PRIMERO QUE APAREZCA) ----------
    candidatos = []

    # estados completos
    for est in estados_con_acento:
        idx = t.find(est)
        if idx != -1:
            candidatos.append((idx, est))

    # abrevs (match por palabra exacta)
    tokens = t.split()
    for tok in tokens:
        if tok in estados_abrev:
            idx = t.find(tok)
            candidatos.append((idx, estados_abrev[tok]))

    if candidatos:
        candidatos.sort(key=lambda x: x[0])  # el primero que aparece
        return pd.Series([None, candidatos[0][1], None])

    return pd.Series([None, "NO ESPECIFICADO", None])

# ================================
# APLICAR AL DATAFRAME Y EXPORTAR
# ================================
df[['MUNICIPIO','ESTADO',"AUTOPISTA"]] = df['LOCATION'].apply(get_municipio_estado)
df["LOCATION"] = df["LOCATION"].apply(clean_keep_accents)
df = df.drop(['MUNICIPIO','AUTOPISTA'], axis=1)
df["ESTADO"] = df["ESTADO"].str.upper().map(estados_con_acento).fillna(df["ESTADO"])
df.loc[df["LOCATION"] == "NO ESPECIFICADO", "ESTADO"] = "NO ESPECIFICADO"
df["ESTADO"]=df["ESTADO"].fillna("NO ESPECIFICADO")
#endregion


### 3.3 Tratamiento de valores duplicados


In [32]:
# ================================
# MANEJO DE LOS DUPLICADOS
# ================================
#region DUPLICADOS
id_col = "CLAIM NUMBER"

def rellenar_informacion_grupo(grupo):
    """
    Completa LOCATION y DESCRIPTION faltantes usando datos del mismo siniestro.
    Ej: Si un registro tiene "Bahía de Acapulco" y otro "NO ESPECIFICADO", 
    copia "Bahía de Acapulco" al registro que lo necesita.
    """
    # Buscar LOCATION válida (no es "NO ESPECIFICADO")
    locations_validas = grupo[grupo["LOCATION"] != "NO ESPECIFICADO"]["LOCATION"].unique()
    if len(locations_validas) > 0:
        location_valida = locations_validas[0]  # Usar la primera disponible
        grupo.loc[grupo["LOCATION"] == "NO ESPECIFICADO", "LOCATION"] = location_valida
    
    # Buscar DESCRIPTION válida
    descriptions_validas = grupo[grupo["DESCRIPTION"] != "NO ESPECIFICADO"]["DESCRIPTION"].unique()
    if len(descriptions_validas) > 0:
        description_valida = descriptions_validas[0]  # Usar la primera disponible
        grupo.loc[grupo["DESCRIPTION"] == "NO ESPECIFICADO", "DESCRIPTION"] = description_valida
    
    # Buscar ESTADO válida
    estados_validos = grupo[grupo["ESTADO"] != "NO ESPECIFICADO"]["ESTADO"].unique()
    if len(estados_validos) > 0:
        estados_validos = estados_validos[0]  # Usar la primera disponible
        grupo.loc[grupo["ESTADO"] == "NO ESPECIFICADO", "ESTADO"] = estados_validos
    
    return grupo

def unir_no_especificado(grupo):
    if len(grupo) == 2:
        fila_buena = grupo[
            (grupo["LOCATION"] != "NO ESPECIFICADO") &
            (grupo["GROSS RESERVE"] > 0) &
            (grupo["DEDUCTIBLE"] > 0) &
            (grupo["DESCRIPTION"] != "NO ESPECIFICADO")
        ]

        # si existe una fila que cumple los criterios, retornarla
        if len(fila_buena) == 1:
            return fila_buena.reset_index(drop=True)

    return grupo

df = (
    df
    .groupby(id_col, group_keys=False)
    .apply(rellenar_informacion_grupo)  # PRIMERO: Rellenar información faltante
    .reset_index(drop=True)
)

df = (
    df
    .groupby(id_col, group_keys=False)
    .apply(unir_no_especificado)  # SEGUNDO: Unir duplicados específicos
    .reset_index(drop=True)
)

df

num_cols = [
    'Cumulative EXPENSES PAID',
    'Cumulative VALUATION EXPENSES',
    'Cumulative CLAIMS PAID',
    'Total Claims Paid Inward',
    'Total Claims Paid no Alae',
    'Inward Incurred',
    'Inward Incurred no Alae',
    'GROSS RESERVE',
    'DEDUCTIBLE'
]

id_col = "CLAIM NUMBER"

def seleccionar_mejores_registros(grupo):

    grupo = grupo.copy()

    # ==============================
    # 0️⃣ PAGOS REALES GANAN TODO
    # ==============================
    # Si hay registros con pagos:
    # - Mantener TODOS esos
    # - PERO también mantener registros SIN pagos si su DEDUCTIBLE > 0 y diferente
    if "Cumulative CLAIMS PAID" in grupo.columns and len(grupo) > 1:
        con_paid = grupo[grupo["Cumulative CLAIMS PAID"].fillna(0) > 0]
        sin_paid = grupo[grupo["Cumulative CLAIMS PAID"].fillna(0) == 0]
        
        if not con_paid.empty and not sin_paid.empty:
            # Verificar si hay DEDUCTIBLE diferentes en registros sin pagos
            # PERO SOLO si el DEDUCTIBLE sin pagos > 0
            ded_con_paid = set(con_paid["DEDUCTIBLE"].fillna(0).unique())
            sin_paid_con_ded_diff = sin_paid[
                (~sin_paid["DEDUCTIBLE"].fillna(0).isin(ded_con_paid)) &
                (sin_paid["DEDUCTIBLE"].fillna(0) > 0)
            ]
            
            # Mantener: registros CON pagos + registros sin pagos con DEDUCTIBLE diferente > 0
            resultado = pd.concat([con_paid, sin_paid_con_ded_diff], ignore_index=True)
            return resultado.reset_index(drop=True)
        elif not con_paid.empty:
            # Solo hay registros con pagos
            return con_paid.reset_index(drop=True)
    
    # ==============================
    # 1️⃣ DEDUCIBLE (duplicados exactos excepto ded)
    # ==============================
    if "DEDUCTIBLE" in grupo.columns and len(grupo) > 1:
        cols_compare = [c for c in grupo.columns if c != "DEDUCTIBLE"]
        dup = grupo.duplicated(subset=cols_compare, keep=False)

        if dup.any():
            candidatos = grupo[dup]
            con_ded = candidatos[candidatos["DEDUCTIBLE"].fillna(0) > 0]

            if not con_ded.empty:
                grupo = con_ded[con_ded["DEDUCTIBLE"] == con_ded["DEDUCTIBLE"].max()]

    # ==============================
    # 2️⃣ DEFINIR FINANZAS (por fila)
    # ==============================
    num_cols = grupo.select_dtypes(include="number").columns
    tiene_finanzas_fila = grupo[num_cols].fillna(0).sum(axis=1) > 0
    ambos_tienen_finanzas = tiene_finanzas_fila.all()

    # ==============================
    # 3️⃣ PAGOS (solo si NO hay reserve/ded)
    # ==============================
    cols_pagos = [
        "Cumulative EXPENSES PAID",
        "Cumulative VALUATION EXPENSES",
        "Cumulative CLAIMS PAID"
    ]
    cols_pagos = [c for c in cols_pagos if c in grupo.columns]

    tiene_reserve_o_ded = (
        (grupo["DEDUCTIBLE"].fillna(0) > 0) |
        (grupo["GROSS RESERVE"].fillna(0) > 0)
    ).any()

    if cols_pagos and len(grupo) > 1 and not tiene_reserve_o_ded:
        tiene_pagos = grupo[cols_pagos].fillna(0).gt(0).any(axis=1)
        if tiene_pagos.any():
            grupo = grupo[tiene_pagos]

    # ==============================
    # 4️⃣ ELIMINAR STATUS = C SIN RESERVE NI DED
    #     (aunque tenga pagos)
    # ==============================
    if len(grupo) > 1 and ambos_tienen_finanzas:
        eliminar = (
            (grupo["GROSS RESERVE"].fillna(0) == 0) &
            (grupo["DEDUCTIBLE"].fillna(0) == 0) &
            (grupo["STATUS"] == "C")
        )

        if eliminar.any() and (~eliminar).any():
            grupo = grupo[~eliminar]

    # ==============================
    # 5️⃣ STATUS P (SOLO si NO ambos tienen finanzas)
    # ==============================
    if "STATUS" in grupo.columns and len(grupo) > 1 and not ambos_tienen_finanzas:
        status_p = grupo[grupo["STATUS"] == "P"]
        if not status_p.empty:
            grupo = status_p

    # ==============================
    # 6️⃣ FILAS CON NÚMEROS
    # ==============================
    if len(num_cols) > 0 and len(grupo) > 1:
        con_numeros = grupo[num_cols].notna().any(axis=1)
        if con_numeros.any():
            grupo = grupo[con_numeros]

    # ==============================
    # 7️⃣ COMPLETITUD
    # ==============================
    completitud = grupo.notna().mean(axis=1)
    grupo = grupo.loc[completitud == completitud.max()]
    
    # ==============================
    # 8️⃣ DESEMPATE FINAL (1 SOLA FILA)
    # ==============================
    if len(grupo) > 1 and len(num_cols) > 0:
    # tomar solo columnas numéricas
      nums = grupo[num_cols].fillna(0)

    # verificar si todas las filas numéricas son iguales
      if nums.nunique().max() == 1:
        # desempate determinístico
        grupo = grupo.sort_index().tail(1)

    return grupo.reset_index(drop=True)

df = (
    df
    .groupby(id_col, group_keys=False)
    .apply(seleccionar_mejores_registros)
    .reset_index(drop=True)
)

df

df.groupby(id_col).size().loc[lambda x: x == 2]
print("Antes:", len(df))
print("Después:", len(df))
claims_con_cero = (
    df.groupby(id_col)
      .apply(lambda g: (g[num_cols].fillna(0).sum(axis=1) == 0).sum() == 1 and len(g) == 2)
)

df["ELIMINADO_POR_CEROS"] = df[id_col].isin(
    claims_con_cero[claims_con_cero].index
)

df=df.drop_duplicates()

df=df.drop(columns="ELIMINADO_POR_CEROS")

#endregion


C:\Users\KOT14\AppData\Local\Temp\ipykernel_30148\2823894913.py:51: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(rellenar_informacion_grupo)  # PRIMERO: Rellenar información faltante
C:\Users\KOT14\AppData\Local\Temp\ipykernel_30148\2823894913.py:58: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(unir_no_especificado)  # SEGUNDO: Unir duplicados específicos
C:\Users\KOT14\AppData\Local\Temp\ipykernel_30

Antes: 4251
Después: 4251


C:\Users\KOT14\AppData\Local\Temp\ipykernel_30148\2823894913.py:213: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g[num_cols].fillna(0).sum(axis=1) == 0).sum() == 1 and len(g) == 2)


## 4. Exportación de resultados


In [33]:
# Guardar Excel
#ruta_salida = os.path.join(carpeta_salida, nombre_archivo_salida)
ruta_salida2 = os.path.join(carpeta_salida2, nombre_archivo_salida)

with pd.ExcelWriter(ruta_salida2, engine="xlsxwriter") as writer:
    df.to_excel(writer, sheet_name="Hoja1", index=False)
    
    workbook  = writer.book
    worksheet = writer.sheets["Hoja1"]
    formato_pct = workbook.add_format({'num_format': '0%'})  # formato porcentaje sin decimales
    col_idx = df.columns.get_loc("KOT SHARE")
    worksheet.set_column(col_idx, col_idx, None, formato_pct)

#print(f"Archivo guardado en: {ruta_salida}")  
print(f"Archivo guardado en: {ruta_salida2}")

Archivo guardado en: C:/Users/KOT14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/202509\202509_Siniestros_Marine_PROCESADO.xlsx
